
# Problem 6 - Policy Iteration on the 5x5 gridworld (EE 5531 Assignment-2).

Alternates two steps until the policy stops changing:

  policy evaluation  : run iterative policy evaluation on the CURRENT policy
  policy improvement : act greedily with respect to the values just computed

The assignment says to reset the values to 0 at the start of every policy
evaluation, so no warm starting between rounds.

Also re-times value iteration here so the comparison in the writeup is
measured on the same machine in the same run.

In [1]:
import time
import numpy as np

In [2]:
import matplotlib
matplotlib.use("Agg")
from Grid_show_actions import plot_actions

In [3]:
GAMMA = 0.95
REWARD = -1.0
TERMINALS = {(0, 0), (4, 4)}
ACTIONS = {"R": (0, 1), "L": (0, -1), "U": (-1, 0), "D": (1, 0)}

In [4]:
def step(row, col, action):
    dr, dc = ACTIONS[action]
    next_row, next_col = row + dr, col + dc
    if next_row < 0 or next_row > 4 or next_col < 0 or next_col > 4:
        return row, col
    return next_row, next_col

In [5]:
def evaluate(policy, gamma=GAMMA, theta=1e-8):
    """Iterative policy evaluation for a deterministic policy. Values start at 0."""
    values = np.zeros((5, 5))
    sweeps = 0

    while True:
        new_values = np.zeros((5, 5))
        for row in range(5):
            for col in range(5):
                if (row, col) in TERMINALS:
                    continue
                next_row, next_col = step(row, col, policy[row][col])
                new_values[row, col] = REWARD + gamma * values[next_row, next_col]

        delta = np.abs(new_values - values).max()
        values = new_values
        sweeps += 1
        if delta < theta:
            return values, sweeps

In [6]:
def improve(values, gamma=GAMMA):
    """Greedy policy w.r.t. values. Ties broken by fixed R, L, U, D order.

    Consistent tie-breaking matters: with an arbitrary rule the policy can
    flip between two equally good actions forever and the stability test
    never fires, even though both policies are optimal.
    """
    policy = [[None] * 5 for _ in range(5)]
    for row in range(5):
        for col in range(5):
            if (row, col) in TERMINALS:
                policy[row][col] = "Nothing"
                continue
            best_action, best_q = None, -np.inf
            for action in ACTIONS:
                next_row, next_col = step(row, col, action)
                q = REWARD + gamma * values[next_row, next_col]
                if q > best_q + 1e-12:      # strict: first action wins a tie
                    best_action, best_q = action, q
            policy[row][col] = best_action
    return policy

In [7]:
def policy_iteration(gamma=GAMMA):
    # Start from an arbitrary deterministic policy: Up everywhere.
    policy = [["Nothing" if (r, c) in TERMINALS else "U" for c in range(5)]
              for r in range(5)]

    total_sweeps = 0
    for round_no in range(1, 100):
        values, sweeps = evaluate(policy, gamma)
        total_sweeps += sweeps
        new_policy = improve(values, gamma)

        changed = sum(new_policy[r][c] != policy[r][c]
                      for r in range(5) for c in range(5))
        print(f"  round {round_no}: {sweeps:3d} evaluation sweeps, "
              f"{changed:2d} states changed action")

        if changed == 0:
            return values, policy, round_no, total_sweeps
        policy = new_policy

    raise RuntimeError("policy iteration did not stabilise")

In [8]:
def value_iteration(gamma=GAMMA, theta=1e-8):
    values = np.zeros((5, 5))
    sweeps = 0
    while True:
        new_values = np.zeros((5, 5))
        for row in range(5):
            for col in range(5):
                if (row, col) in TERMINALS:
                    continue
                new_values[row, col] = max(
                    REWARD + gamma * values[step(row, col, a)]
                    for a in ACTIONS)
        delta = np.abs(new_values - values).max()
        values = new_values
        sweeps += 1
        if delta < theta:
            return values, sweeps

In [9]:
def all_optimal_actions(values, gamma=GAMMA, tol=1e-9):
    """Full tied-action label per state, for the arrow plot."""
    labels = []
    for row in range(5):
        label_row = []
        for col in range(5):
            if (row, col) in TERMINALS:
                label_row.append("Nothing")
                continue
            q = {a: REWARD + gamma * values[step(row, col, a)] for a in ACTIONS}
            best = max(q.values())
            label_row.append("".join(a for a in ACTIONS if q[a] > best - tol))
        labels.append(label_row)
    return labels

In [10]:
def show(values, title):
    print(title)
    for row in range(5):
        print("  " + "  ".join(f"{values[row, col]:8.4f}" for col in range(5)))
    print()

In [11]:
if __name__ == "__main__":
    print("policy iteration (values reset to 0 each evaluation):")
    t0 = time.perf_counter()
    pi_values, pi_policy, rounds, pi_sweeps = policy_iteration()
    pi_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    vi_values, vi_sweeps = value_iteration()
    vi_time = time.perf_counter() - t0

    print()
    show(pi_values, "optimal values from policy iteration:")

    print(f"policy iteration : {rounds} rounds, {pi_sweeps} total sweeps, "
          f"{pi_time * 1000:.2f} ms")
    print(f"value iteration  : {vi_sweeps} sweeps, {vi_time * 1000:.2f} ms")
    print(f"ratio            : {pi_sweeps / vi_sweeps:.1f}x the sweeps, "
          f"{pi_time / vi_time:.1f}x the time\n")

    print(f"max |v_PI - v_VI| = {np.abs(pi_values - vi_values).max():.3e}")

    d = min(1 + 1, 8 - 1 - 1)
    closed = np.array([[-(1 - GAMMA ** min(r + c, 8 - r - c)) / (1 - GAMMA)
                        for c in range(5)] for r in range(5)])
    print(f"max |v_PI - closed form| = {np.abs(pi_values - closed).max():.3e}\n")

    print("deterministic policy returned by policy iteration:")
    for row in pi_policy:
        print("  " + "  ".join(f"{cell:>7}" for cell in row))
    print()

    labels = all_optimal_actions(pi_values)
    print("all optimal actions (for the plot):")
    for row in labels:
        print("  " + "  ".join(f"{cell:>7}" for cell in row))

    assert np.allclose(pi_values, vi_values, atol=1e-6)
    assert np.allclose(pi_values, closed, atol=1e-6)
    print("\nchecks passed: policy iteration and value iteration agree, "
          "both match the closed form")

    plot_actions(labels)
    import matplotlib.pyplot as plt
    plt.savefig("p6_policy.png")
    print("policy plot written to p6_policy.png")

policy iteration (values reset to 0 each evaluation):
  round 1: 361 evaluation sweeps, 19 states changed action
  round 2: 361 evaluation sweeps,  7 states changed action
  round 3: 361 evaluation sweeps,  6 states changed action
  round 4: 361 evaluation sweeps,  2 states changed action
  round 5:   5 evaluation sweeps,  0 states changed action

optimal values from policy iteration:
    0.0000   -1.0000   -1.9500   -2.8525   -3.7099
   -1.0000   -1.9500   -2.8525   -3.7099   -2.8525
   -1.9500   -2.8525   -3.7099   -2.8525   -1.9500
   -2.8525   -3.7099   -2.8525   -1.9500   -1.0000
   -3.7099   -2.8525   -1.9500   -1.0000    0.0000

policy iteration : 5 rounds, 1449 total sweeps, 66.96 ms
value iteration  : 5 sweeps, 0.78 ms
ratio            : 289.8x the sweeps, 86.3x the time

max |v_PI - v_VI| = 0.000e+00
max |v_PI - closed form| = 1.110e-15

deterministic policy returned by policy iteration:
  Nothing        L        L        L        L
        U        L        L        R       

D:\SEM 7\Reinforcement_Learning\Assignment_2\Grid_show_actions.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
